# Görev: Veri Bilimi Hakkında Metin Analizi

Bu örnekte, geleneksel bir veri bilimi sürecinin tüm adımlarını kapsayan basit bir egzersiz yapalım. Hiç kod yazmanız gerekmiyor, aşağıdaki hücrelere tıklayarak onları çalıştırabilir ve sonucu gözlemleyebilirsiniz. Bir zorluk olarak, bu kodu farklı verilerle denemeniz teşvik edilmektedir.

## Amaç

Bu derste, Veri Bilimi ile ilgili çeşitli kavramları tartıştık. Biraz **metin madenciliği** yaparak daha fazla ilgili kavram keşfetmeye çalışalım. Veri Bilimi ile ilgili bir metinle başlayacağız, ondan anahtar kelimeleri çıkaracağız ve ardından sonucu görselleştirmeye çalışacağız.

Metin olarak, Wikipedia'daki Veri Bilimi sayfasını kullanacağım:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Adım 1: Veriyi Almak

Her veri bilimi sürecinin ilk adımı veriyi almaktır. Bunu yapmak için `requests` kütüphanesini kullanacağız:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Adım 2: Veriyi Dönüştürme

Bir sonraki adım, veriyi işlenmeye uygun forma dönüştürmektir. Bizim durumumuzda, sayfadan HTML kaynak kodunu indirdik ve bunu düz metne çevirmemiz gerekiyor.

Bunu yapmanın birçok yolu vardır. HTML ayrıştırma için popüler bir Python kütüphanesi olan [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) kullanacağız. BeautifulSoup, belirli HTML öğelerine odaklanmamıza olanak tanır, böylece Wikipedia'nın ana makale içeriğine odaklanabilir ve bazı gezinme menüleri, yan çubuklar, altbilgiler ve diğer ilgisiz içerikleri azaltabiliriz (bazı şablon metinler yine de kalabilir).


Öncelikle, HTML ayrıştırma için BeautifulSoup kütüphanesini yüklememiz gerekiyor:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Adım 3: İçgörüler Elde Etme

En önemli adım, verilerimizi içgörü elde edebileceğimiz bir forma dönüştürmektir. Bizim durumumuzda, metinden anahtar kelimeleri çıkarmak ve hangi anahtar kelimelerin daha anlamlı olduğunu görmek istiyoruz.

Anahtar kelime çıkarımı için [RAKE](https://github.com/aneesha/RAKE) adlı Python kütüphanesini kullanacağız. Öncelikle, bu kütüphane yüklü değilse yükleyelim:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Ana işlevsellik, bazı parametreler kullanarak özelleştirebileceğimiz `Rake` nesnesinden sağlanır. Bizim durumumuzda, bir anahtar kelimenin minimum uzunluğunu 5 karakter, belgede bir anahtar kelimenin minimum sıklığını 3 ve bir anahtar kelimedeki maksimum kelime sayısını 2 olarak ayarlayacağız. Diğer değerlerle denemeler yapmaktan ve sonucu gözlemlemekten çekinmeyin.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Birlikte ilişkili önem derecesi ile terimler listesini elde ettik. Gördüğünüz gibi, makine öğrenimi ve büyük veri gibi en alakalı disiplinler listede üst sıralarda yer almaktadır.

## Adım 4: Sonucun Görselleştirilmesi

İnsanlar verileri en iyi görsel formda yorumlar. Bu nedenle, bazı içgörüler elde etmek için verileri görselleştirmek sıklıkla mantıklıdır. Anahtar kelimelerin önem derecesi ile basit dağılımını çizmek için Python'da `matplotlib` kütüphanesini kullanabiliriz:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Ancak, kelime frekanslarını görselleştirmenin daha iyi bir yolu var - **Kelime Bulutu** kullanmak. Anahtar kelime listemizden kelime bulutunu çizmek için başka bir kütüphane yüklememiz gerekecek.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` nesnesi, ya orijinal metni ya da önceden hesaplanmış kelimeler ve frekanslarının listesini alır ve ardından `matplotlib` ile görüntülenebilen bir resim döndürür:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Ayrıca orijinal metni `WordCloud`'a geçirebiliriz - benzer bir sonuç alıp alamayacağımıza bakalım:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Kelime bulutunun şimdi daha etkileyici göründüğünü görebilirsiniz, ancak ayrıca birçok gürültü (örneğin `Retrieved on` gibi alakasız kelimeler) içeriyor. Ayrıca, *data scientist* veya *computer science* gibi iki kelimeden oluşan daha az anahtar kelime alıyoruz. Bunun sebebi, RAKE algoritmasının metinden iyi anahtar kelimeleri seçmede çok daha iyi performans göstermesidir. Bu örnek, veri ön işleme ve temizlemenin önemini gösterir; çünkü sonunda elde edilen net resim, daha iyi kararlar almamıza olanak sağlar.

Bu egzersizde, Wikipedia metninden bazı anlamları anahtar kelimeler ve kelime bulutu şeklinde çıkarma sürecinden geçtik. Bu örnek oldukça basittir, ancak bir veri bilimcisinin veriyle çalışırken alacağı tipik adımların tamamını iyi bir şekilde göstermektedir; veri ediniminden görselleştirmeye kadar.

Kursumuzda, bu adımların tamamını detaylı olarak tartışacağız. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
